# Men

In [ ]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings
import importlib

warnings.filterwarnings("ignore")

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything
import xlearner_hill
importlib.reload(xlearner_hill)
get_xlearner = xlearner_hill.get_xlearner

# Tắt bớt log của Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for X-Learner Revenue...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'spend'
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(float)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(float)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(float)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    # Không gian tham số cho LightGBM (dùng chung cho các mô hình con trong X-Learner)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Kích hoạt stochastic path + seed rõ ràng để seed có tác động
        current_params = params.copy()
        current_params['subsample_freq'] = 1
        current_params['bagging_seed'] = SEED
        current_params['feature_fraction_seed'] = SEED
        current_params['data_random_seed'] = SEED
        current_params['verbose'] = -1
        
        # Khởi tạo X-Learner từ file xlearner_hill.py (chỉ định 'revenue')
        x_learner_model = get_xlearner(task_type='revenue', params=current_params, seed=SEED)
        
        # Huấn luyện mô hình X-Learner của EconML
        x_learner_model.fit(Y=y_train, T=t_train, X=X_train)
        
        # Dự đoán Uplift (CATE) trên tập Val bằng hàm .effect()
        uplift_pred_val = x_learner_model.effect(X_val).flatten()
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (X-Learner Robust Revenue)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="X_Learner_Robust_Revenue", sampler=fixed_sampler)
# Có thể giảm n_trials=30 nếu chạy quá lâu
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (REVENUE)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['subsample_freq'] = 1
    best_params_final['bagging_seed'] = SEED
    best_params_final['feature_fraction_seed'] = SEED
    best_params_final['data_random_seed'] = SEED
    best_params_final['verbose'] = -1
    
    # Khởi tạo lại model với best params và seed hiện tại
    final_x_learner = get_xlearner(task_type='revenue', params=best_params_final, seed=SEED)
    
    # Train
    final_x_learner.fit(Y=y_train, T=t_train, X=X_train)
    
    # Predict
    uplift_pred_test = final_x_learner.effect(X_test).flatten()
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình và độ lệch chuẩn
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER REVENUE ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "x_learner_revenue_robust_results_men.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

Best trial: 24. Best value: 0.507748: 100%|██████████| 50/50 [02:21<00:00,  2.83s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.5077
Bộ tham số Robust tốt nhất:
   n_estimators: 96
   learning_rate: 0.004145588003603812
   max_depth: 4
   num_leaves: 126
   min_child_samples: 27
   subsample: 0.9191528315711152
   colsample_bytree: 0.5015553092047234

🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (REVENUE)
Seed 412312: AUUC=0.417, AUQC=0.415, Lift=-0.030, KRCC=-0.025
Seed 42: AUUC=0.417, AUQC=0.416, Lift=0.584, KRCC=-0.070
Seed 1874: AUUC=0.417, AUQC=0.416, Lift=0.586, KRCC=0.044
Seed 902745: AUUC=0.438, AUQC=0.436, Lift=0.266, KRCC=-0.011
Seed 1: AUUC=0.399, AUQC=0.397, Lift=0.360, KRCC=-0.058

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER REVENUE (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.418 ± 0.014
Mean AUQC: 0.416 ± 0.014
Mean Lift: 0.353 ± 0.256
Mean KRCC: -0.024 ± 0.045

💾 Đã lưu kết quả chi tiết từng seed vào: x_learner_revenue_robust_results_men.csv


# Women

In [ ]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings
import importlib

warnings.filterwarnings("ignore")

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything
import xlearner_hill
importlib.reload(xlearner_hill)
get_xlearner = xlearner_hill.get_xlearner

# Tắt bớt log của Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for X-Learner Revenue...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'spend'
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(float)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(float)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(float)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    # Không gian tham số cho LightGBM (dùng chung cho các mô hình con trong X-Learner)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Kích hoạt stochastic path + seed rõ ràng để seed có tác động
        current_params = params.copy()
        current_params['subsample_freq'] = 1
        current_params['bagging_seed'] = SEED
        current_params['feature_fraction_seed'] = SEED
        current_params['data_random_seed'] = SEED
        current_params['verbose'] = -1
        
        # Khởi tạo X-Learner từ file xlearner_hill.py (chỉ định 'revenue')
        x_learner_model = get_xlearner(task_type='revenue', params=current_params, seed=SEED)
        
        # Huấn luyện mô hình X-Learner của EconML
        x_learner_model.fit(Y=y_train, T=t_train, X=X_train)
        
        # Dự đoán Uplift (CATE) trên tập Val bằng hàm .effect()
        uplift_pred_val = x_learner_model.effect(X_val).flatten()
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (X-Learner Robust Revenue)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="X_Learner_Robust_Revenue", sampler=fixed_sampler)
# Có thể giảm n_trials=30 nếu chạy quá lâu
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (REVENUE)")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['subsample_freq'] = 1
    best_params_final['bagging_seed'] = SEED
    best_params_final['feature_fraction_seed'] = SEED
    best_params_final['data_random_seed'] = SEED
    best_params_final['verbose'] = -1
    
    # Khởi tạo lại model với best params và seed hiện tại
    final_x_learner = get_xlearner(task_type='revenue', params=best_params_final, seed=SEED)
    
    # Train
    final_x_learner.fit(Y=y_train, T=t_train, X=X_train)
    
    # Predict
    uplift_pred_test = final_x_learner.effect(X_test).flatten()
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình và độ lệch chuẩn
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER REVENUE ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "x_learner_revenue_robust_results_women.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

Loading data for X-Learner Revenue...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (X-Learner Robust Revenue)...


Best trial: 10. Best value: 1.4831: 100%|██████████| 50/50 [01:58<00:00,  2.37s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 1.4831
Bộ tham số Robust tốt nhất:
   n_estimators: 105
   learning_rate: 0.001059147078541593
   max_depth: 3
   num_leaves: 150
   min_child_samples: 190
   subsample: 0.5091642383955657
   colsample_bytree: 0.5101025466521387

🚀 ĐÁNH GIÁ X-LEARNER BEST PARAMS TRÊN TẬP TEST (REVENUE)
Seed 412312: AUUC=0.779, AUQC=0.777, Lift=0.770, KRCC=0.168
Seed 42: AUUC=0.735, AUQC=0.734, Lift=0.757, KRCC=0.122
Seed 1874: AUUC=0.811, AUQC=0.809, Lift=0.611, KRCC=0.160
Seed 902745: AUUC=0.787, AUQC=0.785, Lift=0.908, KRCC=0.146
Seed 1: AUUC=0.765, AUQC=0.763, Lift=0.715, KRCC=0.137

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH X-LEARNER REVENUE (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.775 ± 0.028
Mean AUQC: 0.774 ± 0.028
Mean Lift: 0.752 ± 0.107
Mean KRCC: 0.147 ± 0.018

💾 Đã lưu kết quả chi tiết từng seed vào: x_learner_revenue_robust_results_women.csv
